# Analyse de la structure des sources de données OSM & DATATourisme

Une fois les sources passées en SILVER, une étape de merge sur un format standardisé est nécessaire afin d'avoir une base unifiée à partir de laquelle créer des itinéraires.

## Install

1. En suivant le lien suivant, on a accès à l'environnement virtuel (.venv) déjà créé pour le projet : </br>https://medium.com/@kishanck/virtual-environments-for-jupyter-notebooks-847b7a3b4da0 </br></br>
    ```sh
    pip3 install ipykernel
    python3 -m ipykernel install --user --name=.venv
    ```
    </br>
2. On reprend manuellement les parties de codes des modules qui nous intéressent

In [2]:
import fastparquet
import geopandas as gpd
import pandas as pd
from shapely import wkb

In [3]:
from pathlib import Path

PARENT_DIR = Path(r'/home/titom/Documents/python_projects/dse_travel_route_opt_app/travel_route_optimization')
DATA_DIR    = PARENT_DIR / "data"
BRONZE_DIR  = PARENT_DIR / "data/bronze"
SILVER_DIR  = PARENT_DIR / "data/silver"
GOLD_DIR    = PARENT_DIR / "data/gold"

OSM_PLACE_NAME              = "Hérault, Occitanie, France"
OSM_BRONZE_DIR              = BRONZE_DIR / "osm"
OSM_BRONZE_GEOPARQUET       = OSM_BRONZE_DIR / "osm_pois.geoparquet"
OSM_BRONZE_GRAPHML          = OSM_BRONZE_DIR / "osm_road_network.graphml"
OSM_SILVER_GEOPARQUET       = SILVER_DIR / "osm_pois.geoparquet"
OSM_SILVER_GRAPHML          = SILVER_DIR / "osm_road_network.graphml"

DT_BRONZE_DIR           = BRONZE_DIR / "datatourisme"
DT_DUMP_URL             = 'https://diffuseur.datatourisme.fr/webservice/e1ec2f4e53628162352a8067eb6ac3e7/071d1b42-f48c-4350-826c-e92199a99bdf'
DT_DUMP_PATH            = DT_BRONZE_DIR / "dt_dump_gz"
DT_DUMP_DIR             = DT_BRONZE_DIR / "dump"
DT_INDEX_FILE           = DT_DUMP_DIR / "index.json"
DT_SILVER_GEOPARQUET    = SILVER_DIR / "datatourisme_pois.geoparquet"
DT_SILVER_CSV           = SILVER_DIR / "datatourisme_pois.csv"   # debug/exploration

In [4]:
def convert_wkb_to_geom(wkb_bytes: bytearray) -> gpd.GeoDataFrame | None: # obligé de passer par là vu que `gpd.read_parquet()` renvoie une erreur
    """Convertit les WKB (byterarray) en shapely.geometry"""
    try:
        return wkb.loads(wkb_bytes)
    except Exception as e:
        return None


def to_geopandas(df: pd.DataFrame) -> gpd.GeoDataFrame: 
    """
    Récupère la géométrie d'un DataFrame.
    Renvoie un GeoDataFrame.
    """
    df['geometry'] = df['geometry'].apply(convert_wkb_to_geom)
    df = df[df['geometry'].notnull()]
    gdf = gpd.GeoDataFrame(df, geometry='geometry')

    return gdf


def get_gdf(geoparquet_path: str) -> gpd.GeoDataFrame:
    df = fastparquet.ParquetFile(geoparquet_path)
    df = df.to_pandas()
    gdf = to_geopandas(df)

    return gdf

In [5]:
gdf_dt = get_gdf(DT_SILVER_GEOPARQUET)

gdf_dt.head()

,id,dc_identifier,source_file,name_fr,name_en,description_fr,types,latitude,longitude,street,...,dept_insee,email,phone,website,opening_hours,allowed_persons,creation_date,last_update,language,geometry
0,https://data.datatourisme.fr/32/395b43c7-f136-...,HLOLAR034V52H27W,https://data.datatourisme.fr/32/395b43c7-f136-...,[FAURE OLIVIER],[FAURE OLIVIER],[Un accueil personnalisé et chaleureux et un p...,schema:Apartment|schema:Product|Accommodation|...,43.447660,3.658580,2 rue du four,...,34,equipage@escondida-croisieres.com,+33 6 79 84 96 56,https://www.escondida-croisieres.com/,2025-01-01 => 2025-12-31,2.0,2024-03-19,2026-02-24,an|es|fr,POINT (3.65858 43.44766)
1,https://data.datatourisme.fr/32/19e0ecd3-a81a-...,HLOLAR034V52RWT4,https://data.datatourisme.fr/32/19e0ecd3-a81a-...,[L'OLIVERAIE À VILLETELLE POUR 6 PERSONNES AVE...,[L'OLIVERAIE À VILLETELLE POUR 6 PERSONNES AVE...,None,Accommodation|PlaceOfInterest|PointOfInterest|...,43.725836,4.132516,1023 Route de Saturargues,...,34,pujolas.location@gmail.com,+33 7 78 20 51 73,None,None,6.0,2025-10-24,2026-02-24,an,POINT (4.13252 43.72584)
2,https://data.datatourisme.fr/32/6bdeee89-f4fb-...,HLOLAR034V52IH8N,https://data.datatourisme.fr/32/6bdeee89-f4fb-...,[STUDIO HUGO],[STUDIO HUGO],"[Studio Hugo est un meublé de 30m2, au rez-de-...",schema:Apartment|schema:Product|Accommodation|...,43.458571,3.424155,12 Rue Victor Hugo,...,34,victorialowe@gmx.co.uk,+33 7 88 60 19 06,None,2025-01-01 => 2025-12-31,NaN,2024-06-04,2026-02-24,an|fr,POINT (3.42415 43.45857)
3,https://data.datatourisme.fr/32/a9fef8fd-334e-...,FMALAR034V52SPYU,https://data.datatourisme.fr/32/a9fef8fd-334e-...,[ATELIER MASQUES],[ATELIER MASQUES],[Venez créer vos masques pour carnaval !],schema:Event|CulturalEvent|EntertainmentAndEve...,43.643264,3.459502,Rue de la Cambalade,...,34,mairieceyras.mediatheque@gmail.com,None,None,2026-03-04 => 2026-03-04,NaN,2026-02-24,2026-02-24,,POINT (3.4595 43.64326)
4,https://data.datatourisme.fr/32/8cc86d1b-76c1-...,HLOLAR034NO09476,https://data.datatourisme.fr/32/8cc86d1b-76c1-...,"[DOMAINE MAS DU PONT - GÎTES, VINS ET TRADITIO...","[DOMAINE MAS DU PONT - GÎTES, VINS ET TRADITIO...",[Proche Camargue et stations balnéaires (Carno...,schema:Apartment|schema:House|schema:Product|A...,43.665763,3.925986,Mas du pont,...,34,gites.masdupont@gmail.com,+33 6 01 19 05 99,https://www.mas-du-pont.com/,None,4.0,2016-03-25,2026-02-24,an,POINT (3.92599 43.66576)


In [9]:
ls_types = set()

for item in gdf_dt['types'].unique():
    ls_type = item.split('|')
    for el in ls_type:
        ls_types.add(el)

print(len(ls_types))
print(ls_types)

167
{'Bakery', 'schema:Event', 'Transporter', 'schema:Hotel', 'schema:SportsEvent', 'LeisureSportActivityProvider', 'MultiPurposeRoomOrCommunityRoom', 'schema:CivicStructure', 'SportsCompetition', 'BusinessPlace', 'Gymnasium', 'Gorge', 'schema:Zoo', 'FoodEstablishment', 'Theater', 'ATM', 'AccommodationProduct', 'ParkAndGarden', 'YouthHostelAndInternationalCenter', 'SelfCateringAccommodation', 'TheaterEvent', 'ActivityProvider', 'Lake', 'Tipi', 'RentalAccommodation', 'HotelRestaurant', 'House', 'ArcheologicalSite', 'schema:House', 'Transport', 'GourmetRestaurant', 'Canal', 'WaterSource', 'schema:FoodEstablishment', 'schema:Hostel', 'HotelTrade', 'ElectricVehicleChargingPoint', 'Arena', 'Accommodation', 'CulturalEvent', 'CulturalActivityProvider', 'Bungalow', 'CaveSinkholeOrAven', 'PlayArea', 'schema:Apartment', 'schema:BusStop', 'CarOrBikeWash', 'ResidentialLeisurePark', 'schema:FastFoodRestaurant', 'CastleAndPrestigeMansion', 'TechnicalHeritage', 'DefenceSite', 'CyclingTour', 'ServiceA

**Categories :**
- "leisure",
- "cultural",
- "historical & religious sites",
- "parks, garden & natural sites",
- "sportive",
- "gastronomy & culinary specialties"
- "restauration"
- "accomodation"
- "transport & mobility"


In [ ]:
{'Bakery',
 'schema:Event',
 'Transporter',
 'schema:Hotel',
 'schema:SportsEvent',
 'LeisureSportActivityProvider',
 'MultiPurposeRoomOrCommunityRoom',
 'schema:CivicStructure',
 'SportsCompetition',
 'BusinessPlace',
 'Gymnasium',
 'Gorge',
 'schema:Zoo',
 'FoodEstablishment',
 'Theater',
 'ATM',
 'AccommodationProduct',
 'ParkAndGarden',
 'YouthHostelAndInternationalCenter',
 'SelfCateringAccommodation',
 'TheaterEvent',
 'ActivityProvider',
 'Lake',
 'Tipi',
 'RentalAccommodation',
 'HotelRestaurant',
 'House',
 'ArcheologicalSite',
 'schema:House',
 'Transport',
 'GourmetRestaurant',
 'Canal',
 'WaterSource',
 'schema:FoodEstablishment',
 'schema:Hostel',
 'HotelTrade',
 'ElectricVehicleChargingPoint',
 'Arena',
 'Accommodation',
 'CulturalEvent',
 'CulturalActivityProvider',
 'Bungalow',
 'CaveSinkholeOrAven',
 'PlayArea',
 'schema:Apartment',
 'schema:BusStop',
 'CarOrBikeWash',
 'ResidentialLeisurePark',
 'schema:FastFoodRestaurant',
 'CastleAndPrestigeMansion',
 'TechnicalHeritage',
 'DefenceSite',
 'CyclingTour',
 'ServiceArea',
 'BusStop',
 'StreetFood',
 'Source',
 'ReligiousSite',
 'LeisureComplex',
 'EquestrianCenter',
 'CamperVanArea',
 'ConvenientService',
 'WalkingTour',
 'Beach',
 'GarageOrAirPump',
 'IceCreamShop',
 'schema:Park',
 'ZooAnimalPark',
 'EquipmentRentalShop',
 'MedicalPlace',
 'NaturalHeritage',
 'Hotel',
 'PointOfView',
 'SportsAndLeisurePlace',
 'BikeStationOrDepot',
 'Producer',
 'schema:MovieTheater',
 'BrasserieOrTavern',
 'Cabaret',
 'PlaceOfInterest',
 'Chalet',
 'schema:Landform',
 'Forest',
 'Room',
 'TreeHouse',
 'schema:BedAndBreakfast',
 'GroupLodging',
 'ClubOrHolidayVillage',
 'Pond',
 'schema:Museum',
 'SelfServiceCafeteria',
 'StopOverOrGroupLodge',
 'RemarkableBuilding',
 'PointOfInterest',
 'ChildrensGite',
 'HolidayResort',
 'KidsClub',
 'ServiceProvider',
 'CafeOrTeahouse',
 'schema:Restaurant',
 'ConventionCentre',
 'schema:Library',
 'Cinema',
 'Apartment',
 'Auditorium',
 'Library',
 'schema:LocalBusiness',
 'RoadTour',
 'schema:Winery',
 'olo:OrderedList',
 'BoutiqueOrLocalShop',
 'PicnicArea',
 'PublicLavatories',
 'Col',
 'FarmhouseInn',
 'WifiHotSpot',
 'Restaurant',
 'FreePractice',
 'schema:CafeOrCoffeeShop',
 'SwimmingPool',
 'Tent',
 'Practice',
 'Museum',
 'CarpoolArea',
 'IceSkatingRink',
 'River',
 'FitnessCenter',
 'BilliardRoom',
 'schema:TheaterEvent',
 'Store',
 'SportsEvent',
 'AccompaniedPractice',
 'TrackRollerOrSkateBoard',
 'TastingProvider',
 'Stables',
 'BarOrPub',
 'TrainStation',
 'Guesthouse',
 'Valley',
 'schema:Bakery',
 'HealthcareProfessional',
 'CollectiveHostel',
 'Product',
 'EntertainmentAndEvent',
 'schema:Product',
 'Cellar',
 'schema:IceCreamShop',
 'CulturalSite',
 'CollectiveAccommodation',
 'Causse',
 'Parking',
 'Traineeship',
 'schema:TrainStation',
 'HolidayCentre',
 'Cirque',
 'UnderwaterRoute',
 'Bog',
 'Waterfall',
 'CampingAndCaravanning',
 'RemembranceSite',
 'BistroOrWineBar',
 'FastFoodRestaurant',
 'Tour',
 'Course',
 'TaxiCompany',
 'Camping',
 'TaxiStation'}